# 개별종목 — LogisticRegression

## 실험 목적

KOSPI200 방향이 상승·보합·하락 중 어디인지 정해졌을 때, 같은 방향일 확률이 높은
개별종목을 찾기 위한 3분류 모델입니다.

후보는 매 거래일 **KOSPI 세부 업종지수 시가총액 상위 10개 × 업종별 KOSPI 보통주
시가총액 상위 5개**로 먼저 고정합니다. 업종의 미래 방향을 따로 예측하는 구조는 아닙니다.

## 공통 조건

| 항목 | 값 |
|---|---|
| 원천 | HF `full/daily_price_dev.parquet`, `full/index_price_dev.parquet` |
| 홀드아웃 | `20240901` 이후 접근 금지 |
| 라벨 | T일 판단 → T+1 `adj_open` 진입 → T+6 `adj_open` 평가, 종목 ±2% |
| 외부 검증 | 날짜 그룹 expanding 12폴드 |
| 최초 학습 | 750거래일 |
| 검증·gap | 폴드당 60거래일 · 직전 5거래일 제거 |
| class weight | 각 외부 폴드 내부에서 `None`과 `balanced` 재비교 |
| 선정 지표 | Accuracy·Macro F1·하락 Recall 조화평균 |

## OOS 결과

| Accuracy | Macro F1 | 하락 Recall | 핵심지표 조화평균 |
|---:|---:|---:|---:|
| 0.4018 | 0.3558 | 0.2582 | **0.3094** |

아래 셀은 저장된 실측 리포트에서 이 모델의 폴드 결과와 class weight 선택 횟수를 다시
읽습니다. 학습 구현은 `models/stock_experiment.py`, 피처·라벨은
`features/stock_model_dataset.py`가 정본입니다.


In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "reports" / "stock_model_experiment.json").exists():
    ROOT = ROOT.parent
report = json.loads((ROOT / "reports" / "stock_model_experiment.json").read_text(encoding="utf-8"))
model_name = 'LogisticRegression'

folds = pd.DataFrame(report["outer_fold_results"])
display(folds.loc[folds["model"].eq(model_name)].reset_index(drop=True))

weights = pd.DataFrame(report["selected_class_weight_counts"])
display(weights.loc[weights["model"].eq(model_name)].reset_index(drop=True))


,model,fold,selected_class_weight,train_dates,valid_dates,train_rows,valid_rows,train_end,valid_start,valid_end,accuracy,macro_f1,down_recall,core_harmonic_mean
0,LogisticRegression,1,balanced,750,60,36207,2880,20130401,20130409,20130704,0.388194,0.340604,0.148421,0.244906
1,LogisticRegression,2,balanced,999,60,48107,2880,20140403,20140411,20140710,0.470486,0.299845,0.102151,0.196721
2,LogisticRegression,3,balanced,1248,60,60029,2880,20150410,20150420,20150715,0.368403,0.368389,0.340245,0.358509
3,LogisticRegression,4,balanced,1497,60,72038,2973,20160414,20160422,20160719,0.425832,0.381046,0.224674,0.318350
4,LogisticRegression,5,balanced,1746,60,84118,2872,20170414,20170424,20170721,0.422702,0.326692,0.176737,0.270640
5,LogisticRegression,6,balanced,1995,60,96025,2940,20180424,20180503,20180731,0.392857,0.387324,0.308308,0.358390
6,LogisticRegression,7,balanced,2243,60,108142,2940,20190502,20190513,20190805,0.426190,0.327570,0.095745,0.189351
7,LogisticRegression,8,balanced,2492,60,120316,2936,20200507,20200515,20200806,0.358311,0.350889,0.546075,0.401497
8,LogisticRegression,9,balanced,2741,60,132514,2955,20210507,20210517,20210809,0.447716,0.398737,0.331073,0.386500
9,LogisticRegression,10,balanced,2990,60,144899,3000,20220511,20220519,20220812,0.359333,0.352474,0.222549,0.296636


,model,selected_class_weight,folds
0,LogisticRegression,balanced,12
